In [5]:
import os
import boto3
import json

# Configure AWS credentials (use env vars AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)
aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
aws_region = 'us-east-1'  # Change to your preferred region

# Create a Bedrock Runtime client
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

# Example 1: Using Claude model
def invoke_claude_model(prompt):
    """
    Invoke Claude model on Amazon Bedrock
    """
    model_id = 'anthropic.claude-3-haiku-20240307-v1:0'
    
    # Prepare the request body
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
    
    # Invoke the model
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(request_body)
    )
    
    # Parse the response
    response_body = json.loads(response['body'].read())
    return response_body['content'][0]['text']


# Example 2: Using Amazon Titan model
def invoke_titan_model(prompt):
    """
    Invoke Amazon Titan model on Bedrock
    """
    model_id = 'amazon.titan-text-express-v1'
    
    request_body = {
        "inputText": prompt,
        "textGenerationConfig": {
            "maxTokenCount": 512,
            "temperature": 0.7,
            "topP": 0.9
        }
    }
    
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(request_body)
    )
    
    response_body = json.loads(response['body'].read())
    return response_body['results'][0]['outputText']


# Example 3: Streaming response with Claude
def invoke_claude_streaming(prompt):
    """
    Invoke Claude model with streaming response
    """
    model_id = 'anthropic.claude-3-5-sonnet-20241022-v2:0'
    
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
    
    response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=model_id,
        body=json.dumps(request_body)
    )
    
    # Process the streaming response
    stream = response.get('body')
    if stream:
        for event in stream:
            chunk = event.get('chunk')
            if chunk:
                chunk_data = json.loads(chunk.get('bytes').decode())
                if chunk_data['type'] == 'content_block_delta':
                    if 'delta' in chunk_data and 'text' in chunk_data['delta']:
                        print(chunk_data['delta']['text'], end='')


# Main execution
if __name__ == "__main__":
    try:
        # Test with Claude
        print("Testing Claude model:")
        prompt = "What is the capital of France?"
        result = invoke_claude_model(prompt)
        print(f"Response: {result}\n")
        
        # Test with streaming
        print("\nTesting Claude with streaming:")
        invoke_claude_streaming("Write a short poem about Python programming.")
        print("\n")
        
    except Exception as e:
        print(f"Error: {str(e)}")

Testing Claude model:
Error: An error occurred (ValidationException) when calling the InvokeModel operation: Operation not allowed


In [6]:
import os
import boto3

aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

regions = ['us-east-1', 'us-west-2', 'eu-central-1']

for region in regions:
    print(f"\n--- Testing region: {region} ---")
    try:
        bedrock = boto3.client(
            service_name='bedrock',
            aws_access_key_id=aws_access_key_id,
            aws_secret_access_key=aws_secret_access_key,
            region_name=region
        )
        
        models = bedrock.list_foundation_models()
        print(f"✓ Can access Bedrock in {region}")
        print(f"  Found {len(models['modelSummaries'])} models")
        
    except Exception as e:
        print(f"✗ Error in {region}: {str(e)}")


--- Testing region: us-east-1 ---
✓ Can access Bedrock in us-east-1
  Found 131 models

--- Testing region: us-west-2 ---
✓ Can access Bedrock in us-west-2
  Found 135 models

--- Testing region: eu-central-1 ---
✓ Can access Bedrock in eu-central-1
  Found 36 models


In [7]:
import os
import boto3
import json

# Configure AWS credentials (use env vars AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)
aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
aws_region = 'us-east-1'  # Mistral is available in us-east-1, us-west-2, eu-west-3

# Create a Bedrock Runtime client
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

def invoke_mistral_large(prompt):
    """
    Invoke Mistral Large model on Amazon Bedrock
    """
    model_id = 'mistral.mistral-large-2402-v1:0'
    
    # Prepare the request body for Mistral
    request_body = {
        "prompt": prompt,
        "max_tokens": 1000,
        "temperature": 0.7,
        "top_p": 0.9
    }
    
    try:
        # Invoke the model
        response = bedrock_runtime.invoke_model(
            modelId=model_id,
            body=json.dumps(request_body)
        )
        
        # Parse the response
        response_body = json.loads(response['body'].read())
        return response_body['outputs'][0]['text']
    
    except Exception as e:
        print(f"Error: {str(e)}")
        raise


def invoke_mistral_large_with_chat(messages):
    """
    Invoke Mistral Large with chat format (multiple messages)
    """
    model_id = 'mistral.mistral-large-2402-v1:0'
    
    # Chat format for Mistral
    request_body = {
        "prompt": format_messages_for_mistral(messages),
        "max_tokens": 1000,
        "temperature": 0.7,
        "top_p": 0.9
    }
    
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(request_body)
    )
    
    response_body = json.loads(response['body'].read())
    return response_body['outputs'][0]['text']


def format_messages_for_mistral(messages):
    """
    Format chat messages for Mistral
    messages: list of dicts with 'role' and 'content'
    Example: [{"role": "user", "content": "Hello"}]
    """
    formatted_prompt = ""
    for msg in messages:
        if msg['role'] == 'user':
            formatted_prompt += f"[INST] {msg['content']} [/INST]"
        elif msg['role'] == 'assistant':
            formatted_prompt += f" {msg['content']}"
    return formatted_prompt


def invoke_mistral_streaming(prompt):
    """
    Invoke Mistral Large with streaming response
    """
    model_id = 'mistral.mistral-large-2402-v1:0'
    
    request_body = {
        "prompt": prompt,
        "max_tokens": 1000,
        "temperature": 0.7
    }
    
    response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=model_id,
        body=json.dumps(request_body)
    )
    
    # Process the streaming response
    stream = response.get('body')
    if stream:
        for event in stream:
            chunk = event.get('chunk')
            if chunk:
                chunk_data = json.loads(chunk.get('bytes').decode())
                if 'outputs' in chunk_data:
                    text = chunk_data['outputs'][0].get('text', '')
                    print(text, end='', flush=True)


# Main execution
if __name__ == "__main__":
    try:
        # Example 1: Simple prompt
        print("Testing Mistral Large model:")
        print("-" * 50)
        prompt = "What is the capital of France? Give a brief answer."
        result = invoke_mistral_large(prompt)
        print(f"Response: {result}\n")
        
        # Example 2: Chat format
        print("\nTesting with chat format:")
        print("-" * 50)
        messages = [
            {"role": "user", "content": "What are the three primary colors?"}
        ]
        result = invoke_mistral_large_with_chat(messages)
        print(f"Response: {result}\n")
        
        # Example 3: Streaming response
        print("\nTesting with streaming:")
        print("-" * 50)
        invoke_mistral_streaming("Write a short haiku about programming.")
        print("\n")
        
    except Exception as e:
        print(f"Error: {str(e)}")

Testing Mistral Large model:
--------------------------------------------------
Error: An error occurred (ValidationException) when calling the InvokeModel operation: Operation not allowed
Error: An error occurred (ValidationException) when calling the InvokeModel operation: Operation not allowed


In [8]:
import os
import boto3
import json

# AWS credentials (use env vars AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)
aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
aws_region = 'us-east-1'

# Create Bedrock client
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

# Invoke Mistral Large model
prompt = "What is the capital of France?"

request_body = {
    "prompt": prompt,
    "max_tokens": 1000,
    "temperature": 0.7
}

response = bedrock_runtime.invoke_model(
    modelId='mistral.mistral-large-2402-v1:0',
    body=json.dumps(request_body)
)

# Get output
response_body = json.loads(response['body'].read())
output = response_body['outputs'][0]['text']

print(output)

ValidationException: An error occurred (ValidationException) when calling the InvokeModel operation: Operation not allowed

In [3]:
# Save as: langgraph_arch_diagrams.py
#
# Install deps first:
#   pip install graphviz
# And ensure system Graphviz is installed (macOS: `brew install graphviz`)
#
# Run:
#   python langgraph_arch_diagrams.py

from graphviz import Digraph

def build_main_graph():
    dot = Digraph("main_graph", format="png")

    # Global styling
    dot.attr(rankdir="LR", splines="ortho", nodesep="1", ranksep="1.3")
    dot.attr("node", shape="box", style="filled", fillcolor="#b0b0b0",
             width="2.2", height="1.0", fontsize="11", fontname="Helvetica")
    dot.attr("edge", arrowsize="0.8", fontsize="9", fontname="Helvetica")

    # Nodes
    dot.node("start", "Start\n(user input)", fillcolor="#d0d0d0")
    dot.node("uie", "User Intent\nEnricher")
    dot.node("router", "Intent Router /\nOrchestrator", fillcolor="#a0a0a0")

    dot.node("analytics_graph", "Analytics\nSubgraph", fillcolor="#c0e0ff")
    dot.node("pd_graph", "Product Detail\nSubgraph", fillcolor="#c0e0ff")
    dot.node("rec_graph", "Recommendation\nSubgraph", fillcolor="#c0e0ff")
    dot.node("oos_graph", "Out Of Scope\nSubgraph", fillcolor="#c0e0ff")

    dot.node("assembler", "Response\nAssembler", fillcolor="#d0d0d0")

    # Edges
    dot.edge("start", "uie", label="State: user_query")
    dot.edge("uie", "router", label="State: +intent")

    dot.edge("router", "analytics_graph", label="if intent == analytics_reporting")
    dot.edge("router", "pd_graph", label="if intent == product_detail")
    dot.edge("router", "rec_graph", label="if intent == recommendation")
    dot.edge("router", "oos_graph", label="fallback / oos")

    # All subgraphs eventually go to assembler with enriched state
    dot.edge("analytics_graph", "assembler",
             label="State: +analytics_insights\n(+product_suggestions?)")
    dot.edge("pd_graph", "assembler",
             label="State: +product_detail")
    dot.edge("rec_graph", "assembler",
             label="State: +recommendations")
    dot.edge("oos_graph", "assembler",
             label="State: +oos_reason")

    # Final answer
    dot.node("end", "Final\nanswer", fillcolor="#d0ffd0")
    dot.edge("assembler", "end", label="compose from state.*")

    return dot


def build_analytics_graph():
    dot = Digraph("analytics_graph", format="png")

    # Global styling
    dot.attr(rankdir="LR", splines="ortho", nodesep="1", ranksep="1.3")
    dot.attr("node", shape="box", style="filled", fillcolor="#b0b0b0",
             width="2.4", height="1.0", fontsize="11", fontname="Helvetica")
    dot.attr("edge", arrowsize="0.8", fontsize="9", fontname="Helvetica")

    # Nodes inside analytics subgraph
    dot.node("entry", "Entry from\nIntent Router",
             fillcolor="#c0e0ff")
    dot.node("ws", "Work Status\nValidator\n(+enrich query)")
    dot.node("analytics", "Analytics\nReporting Agent")
    dot.node("pd_helper", "Product Detail\nHelper\n(uses analytics context)")
    dot.node("assembler", "Analytics\nResponse Assembler",
             fillcolor="#d0d0d0")
    dot.node("exit", "Return to\nMain Graph",
             fillcolor="#c0e0ff")

    # Edges
    dot.edge("entry", "ws",
             label="State: user_query,\nintent == analytics_reporting")
    dot.edge("ws", "analytics",
             label="State: +work_status")

    # Optional path: only if analytics suggests product‑level help
    dot.edge("analytics", "pd_helper",
             label="if analytics_insights\nneeds suggestions")
    dot.edge("analytics", "assembler",
             label="else", style="dashed")

    dot.edge("pd_helper", "assembler",
             label="State: +product_suggestions")

    dot.edge("assembler", "exit",
             label="State: +analytics_insights,\n+product_suggestions?")

    return dot


if __name__ == "__main__":
    main_graph = build_main_graph()
    analytics_subgraph = build_analytics_graph()

    main_path = main_graph.render("main_graph", cleanup=True)
    analytics_path = analytics_subgraph.render("analytics_graph", cleanup=True)

    print(f"Main graph written to: {main_path}")
    print(f"Analytics subgraph written to: {analytics_path}")

ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH